# 05 — FinBERT Sentiment Scoring (Google Colab)

Scores `data/raw/news_headlines.csv` with FinBERT on the GPU and writes
`data/sentiment/{TICKER}_sentiment.csv` for the tickers in `config.yaml`.

**Before running**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `data/raw/news_headlines.csv` to `MyDrive/MARL/data/raw/news_headlines.csv`
3. Push the repo so the clone below has the current `config.yaml` and `src/`

## Step 1 — Check GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > GPU"
print("Device:", torch.cuda.get_device_name(0))

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/MARL"
print("Drive mounted.")

## Step 3 — Clone repo and install

In [ ]:
import os

if not os.path.exists("/content/MARL"):
    !git clone https://github.com/lww637-cmyk/MARL.git /content/MARL
else:
    !git -C /content/MARL pull
%cd /content/MARL
!git log --oneline -1

In [ ]:
!pip install -q -e .
!pip install -q transformers

import sys
sys.path.insert(0, "/content/MARL")
print("Installed.")

## Step 4 — Copy headlines from Drive

In [ ]:
import shutil

os.makedirs("data/raw", exist_ok=True)
shutil.copy(f"{DRIVE}/data/raw/news_headlines.csv", "data/raw/news_headlines.csv")
print("Copied news_headlines.csv")

## Step 5 — Score with FinBERT on GPU, aggregate daily, save
Writes `data/sentiment/{TICKER}_sentiment.csv`.

In [ ]:
from src.data.fetch_sentiment import load_config, score_and_save

config  = load_config()
TICKERS = config["data"]["tickers"]
print("Tickers:", TICKERS)

score_and_save(config, batch_size=128, device=0)

## Step 6 — Copy sentiment files to Drive
Existing files on Drive are kept as `*_sentiment_old.csv`.

In [ ]:
os.makedirs(f"{DRIVE}/data/sentiment", exist_ok=True)
for t in TICKERS:
    dst = f"{DRIVE}/data/sentiment/{t}_sentiment.csv"
    if os.path.exists(dst):
        shutil.copy(dst, f"{DRIVE}/data/sentiment/{t}_sentiment_old.csv")
    shutil.copy(f"data/sentiment/{t}_sentiment.csv", dst)
    print(f"Saved {t} → {dst}")

## Step 7 — Check: headline days per ticker per year
Calendar days with at least one headline (before rolling weekends to trading days).

In [ ]:
import pandas as pd

days = {}
for t in TICKERS:
    df = pd.read_csv(f"data/sentiment/{t}_sentiment.csv", parse_dates=["date"])
    days[t] = df.groupby(df["date"].dt.year).size()

table = pd.DataFrame(days).fillna(0).astype(int)
table.index.name = "year"
print(table.to_string())
print()
for t in TICKERS:
    df = pd.read_csv(f"data/sentiment/{t}_sentiment.csv")
    print(f"{t}: mean score {df['sentiment_score'].mean():+.3f} | "
          f"headlines {int(df['headline_count'].sum())}")